# WEB + SQL — targeted practice drills

Same format as `SS-ADT PRACTICE.ipynb`: question → **write from memory** → check. Reference notebooks: `DB.ipynb`, `WEB DEV.ipynb`.

## Why these drills — your measured weak spots
1. **Implicit edge-case handling** — you lean on `try/except` and loop conditions; here every validation must be an explicit `if`
2. **Exact-name wiring** — `Findvalue` typo, bare `arr` vs `self.__arr`, swapped `update_one` args. Flask punishes this everywhere: form `name=` ↔ `request.form` key ↔ `url_for()` target ↔ route param
3. **Aggregates** — you once wrote `WHERE COUNT(...)`; groups filter with `HAVING`
4. **Parameterisation** — you wrote `LIKE "%Li%"` inline; exam habit is `?` always

Data: `STUDENT.csv` (Student_ID, Student_Name, Class_Group, Handphone), `LATE.csv` (Date, Student_ID, Reason) in this folder.

**Time-boxed. Pick 3–4 drills if tired: 1, 3, 6, 8 give the highest exam payoff.**
Stop the WEB DEV notebook's Flask server before running Drills 6–8 (both use port 5000).

---
## SQL drills

### Drill 1 — Schema + seed from memory (12 min)
Write the full exam loop with **no reference**: connect to `practice.db`, set `row_factory`, DROP both tables, CREATE `Student` and `Late` (composite PK `(Date, Student_ID)` on Late, FK to Student), load both CSVs with `?` placeholders, commit.

**Check (verified):** `SELECT COUNT(*)` → **50** students, **231** lates.

**Must have:** DROP Late before Student (FK order); `next(reader)` for headers; `INSERT OR IGNORE`.

In [20]:
# Drill 1
import sqlite3, csv
conn = sqlite3.connect("practice.db")
conn.row_factory = sqlite3.Row
cur = conn.cursor()


#schema
cur.execute("""
DROP TABLE IF EXISTS Student
""")
cur.execute("""
DROP TABLE IF EXISTS Late
""")
cur.execute("""
CREATE TABLE Student(
stu_id INTEGER PRIMARY KEY,
name TEXT,
civicsclass TEXT,
phonenumber INTEGER
)""")
cur.execute("""
CREATE TABLE Late(
stu_id INTEGER,
date TEXT,
reason TEXT,
PRIMARY KEY(stu_id, date)
)""")

#seed
with open("STUDENT.csv", 'r') as file:
    read = csv.reader(file)
    header = next(read)
    for row in read:
        cur.execute("""
        INSERT OR IGNORE INTO Student(stu_id, name, civicsclass, phonenumber)
        VALUES (?,?,?,?)
        """, (row[0],row[1],row[2],row[3]))
        
with open("LATE.csv", 'r') as file:
    read = csv.reader(file)
    header = next(read)
    for row in read:
        cur.execute("""
        INSERT OR IGNORE INTO Late(date, stu_id, reason)
        VALUES (?,?,?)
        """, (row[0],row[1],row[2]))
        
conn.commit()

#verify
cur.execute("""
SELECT COUNT(*) AS students
FROM Student
""")
print(cur.fetchall()[0]['students'])
cur.execute("""
SELECT COUNT(*) AS latecomings
FROM Late
""")
print(cur.fetchall()[0]['latecomings'])

50
231


#### Safety net
If Drill 1 failed and you want to continue, run the cell below — but note what you got wrong first.

In [3]:
# SAFETY NET — run ONLY if your Drill 1 attempt failed and you need a working DB
import sqlite3, csv

conn = sqlite3.connect('practice.db')
conn.row_factory = sqlite3.Row
cur = conn.cursor()
cur.execute('DROP TABLE IF EXISTS Late')
cur.execute('DROP TABLE IF EXISTS Student')
cur.execute('CREATE TABLE Student(Student_ID INTEGER PRIMARY KEY, Student_Name TEXT, Class_Group TEXT, Handphone INTEGER)')
cur.execute('CREATE TABLE Late(Date TEXT, Student_ID INTEGER, Reason TEXT, PRIMARY KEY (Date, Student_ID), FOREIGN KEY (Student_ID) REFERENCES Student(Student_ID))')
for fname, table, n in (('STUDENT.csv', 'Student', 4), ('LATE.csv', 'Late', 3)):
    with open(fname) as f:
        reader = csv.reader(f)
        next(reader)
        marks = ','.join('?' * n)
        for row in reader:
            cur.execute(f'INSERT OR IGNORE INTO {table} VALUES ({marks})', row)
            #this is why claude is the goat
conn.commit()
cur.execute('SELECT COUNT(*) FROM Student'); print('students:', cur.fetchone()[0])
cur.execute('SELECT COUNT(*) FROM Late');    print('lates:   ', cur.fetchone()[0])

#comments from after completing first cell: this insertion is simply way more optimal. i need to memorise this. 

students: 50
lates:    231


### Drill 2 — Parameterised query trio (8 min)
All three on one connection, **every value through `?`**, none hardcoded:

**(a)** Names + dates of lates for a class typed via `input()`, newest date first.
**(b)** All lates whose `Reason` contains the word `bus` — `LIKE` with the wildcard string passed as a parameter, not spliced into the SQL.
**(c)** Name and handphone of one student by `Student_ID` — use `fetchone()`.

**Trap being drilled:** `cur.execute("... LIKE ?", ("%bus%",))` — the `%` lives in the Python tuple, never in the SQL string.

**Check (verified):** (b) returns **100** rows.

In [22]:
# Drill 2

def find_late_namesndate(civicsclass):
    cur.execute("""
    SELECT Student.name, Late.date
    FROM Student INNER JOIN Late ON Student.stu_id = Late.stu_id
    WHERE Student.civicsclass = ?
    ORDER BY Late.date DESC;
    """, (civicsclass,))
    return cur.fetchall()
    
    
for row in find_late_namesndate(input("enter civics class: ")):
    print(row["name"], row["date"])
    
#skipping the output for the rest of the subtasks to avoid flooding cell output

cur.execute("""
SELECT *
FROM Late
WHERE reason LIKE ?;
""", ("%bus%",))

def find_student_namenphone(student_id):
    cur.execute("""
    SELECT name, phonenumber
    FROM Student
    WHERE stu_id = ?;
    """, (student_id))
    return cur.fetchone()

enter civics class: 25S02X
Philippe de la Cruz 2024-09-30
Indira Devi 2024-09-27
Mohammad Rizwan 2024-09-27
Dylan Koh 2024-09-26
Lee Xue Ying 2024-09-24
Sara Tay 2024-09-24
Amy Lim 2024-09-20
Anh Tran 2024-09-18
Nguyen Thi Mai 2024-09-17
Chen Wei Li 2024-09-17
Natasha Lim 2024-09-16
Prakash Ravi 2024-09-16
Boon Kai Sheng 2024-09-15
Nguyen Hoang 2024-09-13
Kumar Suresh 2024-09-11
Chen Wei Li 2024-09-10
Indira Devi 2024-09-09
Siti Nurhaliza 2024-09-08
Amy Lim 2024-08-27
Tanisha Pillai 2024-08-26
Sara Tay 2024-08-26
Dylan Koh 2024-08-23
Anh Tran 2024-08-22
Tay Mingxuan Cory 2024-08-22
Prakash Ravi 2024-08-21
Chen Wei Li 2024-08-21
Boon Kai Sheng 2024-08-20
Nguyen Thi Mai 2024-08-19
Siti Nurhaliza 2024-08-19
Kumar Suresh 2024-08-18
Indira Devi 2024-08-15
Kim Choon Yee 2024-08-14
Nguyen Hoang 2024-08-13
Natasha Lim 2024-08-08
Ali Faisal 2024-08-06
Lydia Tan 2024-08-06
Philippe de la Cruz 2024-08-05
Kim Choon Yee 2024-08-04
Siti Nurhaliza 2024-08-01
Chen Wei Li 2024-08-01
Tay Mingxuan Cory 2

### Drill 3 — GROUP BY / HAVING (8 min)
**(a)** Each student's name and late count, **2 or more only**, biggest first.
**(b)** Each class and its late count — but only counting lates with `Reason` containing `bus`. Think: does that condition go in `WHERE` or `HAVING`, and why?

**Check (a, verified):** top three rows are Boon Kai Jie 10, Liu Ying 10, Joshua Ong 8.
**Answer (b):** row condition → `WHERE`; it filters individual lates *before* grouping. `HAVING` is only for conditions **on the aggregate itself**.

In [ ]:
# Drill 3

cur.execute("""
SELECT Student.name, COUNT(*) AS latecount
FROM Student INNER JOIN Late ON Student.stu_id = Late.stu_id
GROUP BY Late.stu_id
HAVING latecount >= 2
ORDER BY latecount DESC
""")

cur.execute("""
SELECT Student.civicsclass, COUNT(*) AS latecount 
FROM Student INNER JOIN Late ON Student.stu_id = Late.stu_id
WHERE Late.reason LIKE ?
GROUP BY Student.civicsclass
""", ("%bus%", ))


### Drill 4 — Never-late list (5 min)
Prompt for a class; print students in that class with **zero** late records.

**Check:** class `25S02X` → Tan Yu Wei, Sim Wen Jie, Rina Ong.

**Pattern:** LEFT OUTER JOIN, then `WHERE Late.Student_ID IS NULL`.

In [ ]:
# Drill 4

def never_late(civicsclass):
    cur.execute("""
    SELECT Students.name 
    FROM Student LEFT OUTER JOIN Late ON Student.stu_id = Late.stu_id
    WHERE Late.stu_id IS NULL AND Students.civicsclass LIKE ?
    """, (f"%{civicsclass}%",))

### Drill 5 — SQL debug (8 min)
Each snippet has exactly one conceptual bug. Fix in place and state the rule it violates.

In [ ]:
# Drill 5 — fix each

# (a) students with more than 2 lates
cur.execute('''
SELECT Student.Student_Name
FROM Student INNER JOIN Late ON Student.Student_ID = Late.Student_ID
GROUP BY Student.Student_ID
HAVING COUNT(*) > 2
''') #using aggregate function COUNT before GROUP BY

# (b) search by reason typed by the user
reason = input("reason: ")
cur.execute(f"SELECT * FROM Late WHERE Reason LIKE '%{reason}%'")
#will not return anything if input does not match reason text exactly

# (c) class, reason, and late count per class
cur.execute('''
SELECT Student.Class_Group, Late.Reason, COUNT(*) AS latecount
FROM Student INNER JOIN Late ON Student.Student_ID = Late.Student_ID
GROUP BY Student.Class_Group
''') #did not add table before quoting the field 

#### Drill 5 answers — check only after attempting
**(a)** Aggregates cannot go in `WHERE` — move to `HAVING COUNT(*) > 2` after the `GROUP BY`.
**(b)** F-string splicing = injection + quoting bugs. `cur.execute("SELECT * FROM Late WHERE Reason = ?", (reason,))`.
**(c)** `Reason` is neither grouped nor aggregated — after `GROUP BY Class_Group` each output row is a whole class, so a single `Reason` is meaningless. Drop it, add it to the `GROUP BY`, or aggregate it (`COUNT(DISTINCT Reason)`).

---
## Flask drills

Templates already exist on disk (`templates/home.html`, `results.html`, `add.html`, `static/style.css`) — written by `WEB DEV.ipynb`. These drills test whether you can wire an app to them **from memory**. The wiring facts you must recall, not look up:

- `home.html` posts to `/` with fields **`category`** and **`name`**
- `results.html` loops `results` rows using keys like `row['Student_Name']`, and shows `{{ name }}` / `{{ category }}`
- `add.html` posts to **`url_for('add_late')`** with fields `stu_id`, `date`, `reason`

### Drill 6 — Search flow from memory (15 min)
Write the app: GET `/` renders the form; POST `/` reads `category` + `name`, **explicitly** validates both non-empty (an `if`, not a try/except), re-renders with `error=` if not, else `redirect(url_for(...))` to `/results/<category>/<name>`; the results route opens its own connection, JOINs Student–Late filtered by reason and partial name match, closes, renders.

Run, open `http://127.0.0.1:5000`, search `amy lim` / `Overslept` — expect at least one row. Interrupt the kernel to stop the server before moving on.

In [ ]:
# Drill 6

### Drill 7 — Insert flow from memory (10 min)
Add the `/add` route to your Drill 6 app (or a fresh app).

- GET renders `add.html`
- POST: read `stu_id`, `date`, `reason`; **explicit `if`** rejects any empty field with `error=`; valid → parameterised INSERT, commit, `redirect(url_for("home"))`

**The trap this drill exists for:** `add.html`'s form action is `url_for('add_late')` — your view function **must be named `add_late`**, or Jinja raises BuildError the moment the page renders. Name-wiring again.

In [ ]:
# Drill 7

### Drill 8 — Flask debug (10 min)
Six planted bugs — every one is a mistake class you have actually made this week. Fix in place.

In [ ]:
# Drill 8 — fix all SIX bugs in place, then explain each in one line
from flask import Flask, render_template, request, redirect, url_for
import sqlite3

app = Flask(__name__)

@app.route("/", methods = ['GET', 'POST'])                                     # bug 1 route missing accepted methods
def home():
    if request.method == "POST":
        category = request.form.get("category")       # bug 2 spelling error
        name = request.form.get("name")
        if name and category:
            return redirect(url_for("results", category=category, name=name))   # bug 3 need to return a redirect/render
    return render_template("home.html")

@app.route("/results/<category>/<name>", methods = ['GET', 'POST'])
def result(category, name):                          # bug 4 ??tm
    conn = sqlite3.connect("practice.db")
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    cur.execute("SELECT * FROM Late WHERE Reason LIKE ?", (f"%{category}%",))       # bug 5 fstring injection
    data = cur.fetchall()
    conn.close()
    return render_template("results.html", results=data, category=category, name=name)

# bug 6: what row_factory line is missing for results.html to use row['Student_Name']?
# conn.row_factory = sqlite3.Row is missing.

# if __name__ == "__main__":
#     app.run(port=5000)      # uncomment to test after fixing

#### Drill 8 answers — check only after attempting
1. Route handles POST but declares no methods → `@app.route("/", methods=["GET", "POST"])`
2. `"catgory"` ≠ the form's `name="category"` → `.get()` silently returns `None` — misspelt keys don't crash, they just break logic
3. **Missing `return`** before `redirect(...)` — the redirect object is created and discarded, page just re-renders (your Drill 1 binary-search bug, Flask edition)
4. Function named `result` but `url_for("results", ...)` targets `results` → BuildError at runtime
5. F-string SQL splice → `cur.execute("SELECT * FROM Late WHERE Reason = ?", (category,))`
6. `conn.row_factory = sqlite3.Row` missing → `row['Student_Name']` in the template raises; without it rows are plain tuples

---
## Quick self-mark

| Drill | Pass if |
|-------|---------|
| 1 | tables build fresh on re-run; composite PK + FK present; counts print |
| 2 | zero literals inside SQL strings; `%bus%` passed as a parameter |
| 3 | (a) uses HAVING; (b) you put the reason filter in WHERE and can say why |
| 4 | the three never-late names for 25S02X appear |
| 5 | three rules stated, not just code edited |
| 6 | full round trip in the browser: form → redirect → results table |
| 7 | empty field re-renders with error; valid insert lands in DB and redirects |
| 8 | all six found — especially the missing `return` (bug 3) |

### The one habit that fixes most of these
Before running anything, scan every name pair and ask "do these match, character for character?" — form `name=` vs `request.form` key, `url_for()` string vs function name, route `<param>` vs function argument, SQL column vs template key. Your recurring bug class is not logic — it is **wiring**.